# 01 — Test Configuration & First GRPO Training

Questo notebook verifica passo-passo che tutto il setup funzioni:

1. **Imports & GPU** — verifica che torch veda la GPU
2. **Dataset** — genera (o carica) il dataset sintetico (solo JSON)
3. **Model** — carica il modello con LoRA + 4-bit
4. **Rewards** — test rapido delle reward functions (JSON)
5. **Inference test** — genera un paio di completions e valuta
6. **GRPO Training** — avvia un mini-train (pochi step)

## Setup su Colab

In [1]:
!git clone https://github.com/GiuseppeBellamacina/tris.git
%cd tris
!bash setup.sh

Cloning into 'tris'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 148 (delta 61), reused 128 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 315.89 KiB | 1.83 MiB/s, done.
Resolving deltas: 100% (61/61), done.
/content/tris
=== Setup GRPO Strict Generation (Colab) ===

📦 Installazione uv...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 81.7 MB/s eta 0:00:00:00:0100:01
✅ uv installato

📦 Installazione dipendenze + progetto...
Using Python 3.12.13 environment at: /usr
Resolved 282 packages in 1.67s                                       
Prepared 72 packages in 17.08s                                           
Uninstalled 10 packages in 559ms
Installed 72 packages in 519ms                              
 + anthropic==0.86.0
 + apache-tvm-ffi==0.1.9
 + astor==0.8.1
 + async-lru==2.3.0
 + bitsandbytes==0.49.2
 + blake3==1.0.8
 + cbor2==5.9.0
 + compress

## 1. Imports & GPU Check

In [2]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(
        f"VRAM:            {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
    )
else:
    print("WARNING: Nessuna GPU trovata! Il training sarà molto lento.")

PyTorch version: 2.10.0+cu128
CUDA available:  True
GPU:             Tesla T4
VRAM:            15.6 GB


In [3]:
import accelerate
import peft
import transformers
import trl

import datasets as hf_datasets

print(f"transformers: {transformers.__version__}")
print(f"trl:          {trl.__version__}")
print(f"peft:         {peft.__version__}")
print(f"datasets:     {hf_datasets.__version__}")
print(f"accelerate:   {accelerate.__version__}")
print("\nTutti gli import OK")

transformers: 4.57.6
trl:          0.24.0
peft:         0.18.1
datasets:     4.3.0
accelerate:   1.13.0

Tutti gli import OK


## 2. Genera / Carica Dataset Sintetico

In [4]:
import os
from pathlib import Path

from src.datasets.dataloader import load_synthetic_dataset
from src.datasets.synthetic_dataset import generate_dataset

ROOT = Path.cwd()  # la root del progetto è la cartella corrente

os.chdir(ROOT)  # assicurati di essere nella root

DATASET_PATH = ROOT / "data" / "synthetic"

if not DATASET_PATH.exists():
    print("Dataset non trovato, lo genero...")
    ds = generate_dataset(num_samples=500, seed=42)  # 500 per test veloce
    ds.save_to_disk(str(DATASET_PATH))
    print(f"Dataset salvato in {DATASET_PATH}")
else:
    print(f"Dataset trovato in {DATASET_PATH}")

ds = load_synthetic_dataset(str(DATASET_PATH))
print(f"\nSplits: {list(ds.keys())}")
for split_name, split_ds in ds.items():
    print(f"  {split_name}: {len(split_ds)} samples")
    print(f"    columns: {split_ds.column_names}")

Dataset non trovato, lo genero...


Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

Dataset salvato in /content/tris/data/synthetic

Splits: ['train', 'test']
  train: 400 samples
    columns: ['system_prompt', 'prompt', 'task_type', 'difficulty']
  test: 100 samples
    columns: ['system_prompt', 'prompt', 'task_type', 'difficulty']


In [5]:
# Guarda qualche sample
train_ds = ds["train"]
for i in range(3):
    sample = train_ds[i]
    print(f"--- Sample {i} ---")
    print(f"  difficulty: {sample['difficulty']}")
    print(f"  prompt:     {sample['prompt'][:120]}...")
    print()

--- Sample 0 ---
  task_type:  python
  difficulty: hard
  prompt:     Write a Python context manager class called `DatabaseTransaction` that measures and stores the elapsed time of the code ...

--- Sample 1 ---
  task_type:  json
  difficulty: hard
  prompt:     Generate a deeply nested JSON object (at least 4 levels of nesting) representing a file system directory tree hierarchy....

--- Sample 2 ---
  task_type:  python
  difficulty: simple
  prompt:     Write a Python function `celsius_to_fahrenheit` that converts a string to a binary string representation....



## 3. Carica Modello + LoRA + 4-bit

In [6]:
from src.utils.config import load_config

config = load_config(str(ROOT / "experiments" / "configs" / "grpo.yaml"))
print("Config caricata:")
for section, values in config.items():
    print(f"  [{section}]")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"    {k}: {v}")

Config caricata:
  [model]
    name: Qwen/Qwen2.5-Coder-1.5B-Instruct
    quantization: 4bit
    dtype: bfloat16
    use_unsloth: False
  [lora]
    r: 16
    lora_alpha: 32
    lora_dropout: 0.05
    target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj']
    task_type: CAUSAL_LM
  [dataset]
    path: data/synthetic
    split: train
    max_samples: None
  [training]
    max_steps: 1000
    per_device_train_batch_size: 1
    gradient_accumulation_steps: 8
    learning_rate: 1e-06
    lr_scheduler_type: cosine
    warmup_steps: 50
    bf16: True
    logging_steps: 10
    save_steps: 200
    save_total_limit: 3
    output_dir: experiments/checkpoints/grpo
    log_dir: experiments/logs/grpo
  [grpo]
    num_generations: 8
    max_completion_length: 512
    max_prompt_length: 256
    beta: 0.04
    temperature: 0.7
    use_vllm: False
  [reward]
    type: combined
    partial_credit: False
    reasoning_bonus: 0.0
  [wandb]
    project: grpo-strict-generation
    run_name: None
    tags:

In [7]:
from src.models.model_loader import load_model_and_tokenizer

print(f"Caricamento modello: {config['model']['name']}")
print("(questo può richiedere qualche minuto al primo download...)\n")

model, tokenizer = load_model_and_tokenizer(config)

# Info modello
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nParametri totali:    {total:,}")
print(f"Parametri trainable: {trainable:,} ({100 * trainable / total:.2f}%)")
print(f"Tokenizer vocab:     {len(tokenizer)}")
print(
    f"Pad token:           {tokenizer.pad_token} (id={tokenizer.pad_token_id})"
)

Caricamento modello: Qwen/Qwen2.5-Coder-1.5B-Instruct
(questo può richiedere qualche minuto al primo download...)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Parametri totali:    892,974,592
Parametri trainable: 4,358,144 (0.49%)
Tokenizer vocab:     151665
Pad token:           <|endoftext|> (id=151643)


## 4. Test Reward Functions

In [8]:
from src.training.rewards import (
    combined_reward,
    json_reward,
    reasoning_reward,
)

# Test JSON reward
valid_json = '```json\n{"name": "test", "age": 25, "active": true}\n```'
invalid_json = '```json\n{"name": "test", age: 25}\n```'
no_json = "Non ho generato nessun JSON."

print("=== JSON Reward ===")
print(f"  Valid JSON:   {json_reward(valid_json)}")  # atteso: 1.0
print(f"  Invalid JSON: {json_reward(invalid_json)}")  # atteso: 0.0
print(f"  No JSON:      {json_reward(no_json)}")  # atteso: 0.0

# Test con partial credit
print("\n=== JSON Reward (partial credit) ===")
print(
    f"  Valid JSON:   {json_reward(valid_json, partial_credit=True)}"
)  # atteso: 1.0
print(
    f"  Invalid JSON: {json_reward(invalid_json, partial_credit=True)}"
)  # atteso: 0.0 o 0.5

# Test reasoning bonus
with_think = (
    "<think>Devo creare un JSON con tre campi come richiesto.</think>\n"
    + valid_json
)
without_think = valid_json

print("\n=== Reasoning Reward ===")
print(f"  With <think>:    {reasoning_reward(with_think)}")  # atteso: 0.2
print(f"  Without <think>: {reasoning_reward(without_think)}")  # atteso: 0.0

# Test combined (solo JSON)
print("\n=== Combined Reward (JSON) ===")
print(f"  JSON valid:           {combined_reward(valid_json, 'json')}")  # 1.0
r = combined_reward(with_think, "json", reasoning_bonus=0.2)
print(f"  JSON valid + reason:  {r}")  # 1.2
print(f"  JSON invalid:         {combined_reward(invalid_json, 'json')}")  # 0.0

print("\nReward functions OK")

=== JSON Reward ===
  Valid JSON:   1.0
  Invalid JSON: 0.0
  No JSON:      0.0

=== Python Reward ===
  Valid Python:   1.0
  Invalid Python: 0.0
  No Python:      0.0

=== Reasoning Reward ===
  With <think>:    0.2
  Without <think>: 0.0

=== Combined Reward ===
  JSON valid:           1.0
  JSON valid + reason:  1.2

Reward functions OK


## 5. Test Inference (pre-training)

In [9]:
from src.datasets.dataloader import format_prompt_for_model

# Prendi 4 sample (tutti JSON ora)
test_indices = [0, 1, 10, 20]
test_samples = [train_ds[i] for i in test_indices if i < len(train_ds)]

print(f"Genero completions per {len(test_samples)} prompt...\n")

for sample in test_samples:
    prompt = format_prompt_for_model(sample, tokenizer)
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    completion = tokenizer.decode(
        outputs[0][input_len:], skip_special_tokens=True
    )

    # Valuta (tutti JSON)
    difficulty = sample["difficulty"]
    reward = combined_reward(completion)

    print(f"--- [json/{difficulty}] ---")
    print(f"Prompt:     {sample['prompt'][:100]}...")
    print(f"Completion: {completion[:200]}")
    print(f"Reward:     {reward}")
    print()

Genero completions per 4 prompt...

--- [python/hard] ---
Prompt:     Write a Python context manager class called `DatabaseTransaction` that measures and stores the elaps...
Completion: ```python
import time

class DatabaseTransaction:
    def __init__(self, connection):
        self.connection = connection
        self.start_time = None

    def __enter__(self):
        try:
       
Reward:     1.0

--- [json/hard] ---
Prompt:     Generate a deeply nested JSON object (at least 4 levels of nesting) representing a file system direc...
Completion: ```json
{
  "files": [
    {
      "id": 1,
      "name": "root",
      "children": [
        {
          "id": 2,
          "name": "documents",
          "children": [
            {
              "i
Reward:     0.0

--- [json/medium] ---
Prompt:     Generate a JSON object representing a employee record with at least 6 fields, including one nested o...
Completion: ```json
{
  "employeeId": 12345,
  "firstName": "John",
  "lastName": "Doe",
  "

## 6. Mini GRPO Training (pochi step di test)

Facciamo un training ridotto per verificare che la pipeline giri senza errori.

- **max_steps=20** — solo per test
- **num_generations=4** — minimo per avere varianza nelle reward
- **learning_rate=5e-6** — come da notebook ufficiali Unsloth
- **beta=0.1** — KL penalty più alto per restare vicini al modello base
- **optim=paged_adamw_8bit** — meno VRAM
- **50 samples** — subset piccolo

### Logging con wandb
Il training logga metriche (reward, KL, loss) su [Weights & Biases](https://wandb.ai).
Su Colab la prima volta chiede l'API key (prendi da https://wandb.ai/authorize).
Per saltare wandb, imposta `report_to="none"` nella cella GRPOConfig.

In [ ]:
import wandb

# Login a wandb — necessario su Colab (la prima volta chiede l'API key).
# Prendi la tua key da https://wandb.ai/authorize
# Se vuoi saltare wandb, imposta report_to="none" nella cella GRPOConfig.
wandb.login()

In [ ]:
from datasets import Dataset
from src.training.rewards import build_reward_function

# Usa solo 50 samples per il test
MINI_N = 50
mini_ds = train_ds.select(range(min(MINI_N, len(train_ds))))  # type: ignore[arg-type]

# Formatta i prompt
formatted = []
for i in range(len(mini_ds)):
    s = mini_ds[i]
    p = format_prompt_for_model(s, tokenizer)
    formatted.append({"prompt": p})

prompt_dataset = Dataset.from_list(formatted)
print(f"Mini dataset: {len(prompt_dataset)} prompt pronti")

# Reward function
reward_fn = build_reward_function(config["reward"])
print("Reward function creata")

In [ ]:
import datetime

from trl import GRPOConfig

# Config leggera per test
mini_output_dir = str(ROOT / "experiments" / "checkpoints" / "grpo_test")
mini_log_dir = str(ROOT / "experiments" / "logs" / "grpo_test")
Path(mini_output_dir).mkdir(parents=True, exist_ok=True)
Path(mini_log_dir).mkdir(parents=True, exist_ok=True)

_ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"grpo-test-hf-{_ts}"

# bf16 richiede GPU Ampere+ (compute capability >= 8.0);
# su GPU più vecchie (es. T4/V100) si ricade su fp16.
_supports_bf16 = (
    torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
)
use_bf16 = _supports_bf16
use_fp16 = not _supports_bf16 and torch.cuda.is_available()
print(
    f"bf16={use_bf16}, fp16={use_fp16}  "
    f"(GPU compute capability: {torch.cuda.get_device_capability() if torch.cuda.is_available() else 'N/A'})"
)

grpo_config = GRPOConfig(  # type: ignore[call-arg]
    output_dir=mini_output_dir,
    run_name=run_name,
    # Training ridotto
    max_steps=20,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    max_grad_norm=0.1,
    bf16=use_bf16,
    fp16=use_fp16,
    # GRPO
    num_generations=4,
    max_completion_length=256,
    max_prompt_length=256,
    beta=0.1,
    temperature=0.7,
    # logging
    logging_dir=mini_log_dir,
    logging_steps=1,
    report_to="wandb",
)

print("GRPOConfig creata")
print(f"  run_name:           {grpo_config.run_name}")
print(f"  max_steps:          {grpo_config.max_steps}")
print(f"  num_generations:    {grpo_config.num_generations}")
print(f"  batch_size:         {grpo_config.per_device_train_batch_size}")
print(f"  grad_accum:         {grpo_config.gradient_accumulation_steps}")
print(f"  learning_rate:      {grpo_config.learning_rate}")
print(f"  optim:              {grpo_config.optim}")
print(f"  beta (KL penalty):  {grpo_config.beta}")
print(f"  logging_steps:      {grpo_config.logging_steps}")
print(f"  report_to:          {grpo_config.report_to}")

In [ ]:
from trl import GRPOTrainer

# Ricarica il modello fresco
torch.cuda.empty_cache()
model, tokenizer = load_model_and_tokenizer(config)

trainer = GRPOTrainer(  # type: ignore[arg-type]
    model=model,
    args=grpo_config,
    train_dataset=prompt_dataset,
    reward_funcs=reward_fn,
    processing_class=tokenizer,
)

print("GRPOTrainer inizializzato")
print(f"  Tipo effettivo: {type(trainer).__name__}")
print("\nAvvio mini-training (20 step)...\n")

trainer.train()
print("\nMini-training completato")

## 7. Verifica post-training

In [ ]:
# Rigenera sugli stessi prompt di prima per vedere se qualcosa è cambiato
print("Completions dopo il mini-training:\n")

for sample in test_samples:
    prompt = format_prompt_for_model(sample, tokenizer)
    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    input_len = inputs["input_ids"].shape[1]
    completion = tokenizer.decode(
        outputs[0][input_len:], skip_special_tokens=True
    )

    difficulty = sample["difficulty"]
    reward = combined_reward(completion)

    print(f"--- [json/{difficulty}] ---")
    print(f"Prompt:     {sample['prompt'][:100]}...")
    print(f"Completion: {completion[:200]}")
    print(f"Reward:     {reward}")
    print()

In [ ]:
# Pulizia VRAM
del trainer
del model
torch.cuda.empty_cache()
print("VRAM liberata")

**Accelerazione con Unsloth:**
- **Unsloth** (`use_unsloth: true`): 2-5x training più veloce, ~50-70% meno VRAM
- **vLLM**: disabilitato (richiede server separato, non disponibile su Colab)
- Installa con: `uv pip install -e ".[fast]"`

**Focus: solo generazione JSON** — il dataset contiene esclusivamente task JSON.

**Prossimi passi:**
- Training completo: `uv run python -m src.training.grpo_train --config experiments/configs/grpo.yaml`
- Baseline eval: `uv run python -m src.evaluation.baseline_eval --config experiments/configs/baseline.yaml`
- SFT comparison: `uv run python -m src.training.sft_train --config experiments/configs/sft.yaml`